# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ARTI-INTEL/fr-ml-intern-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

**Lane 2 — Refresh / Content Opportunity Scoring**

This notebook builds the full feature vector, audits every feature for leakage (label-derived, future-window, product-flag), and compares random-split vs grouped-split validation. Every claim below has a code cell next to it.

---
## Setup — connect to the warehouse

In [1]:
import duckdb, os, getpass, warnings
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import classification_report, roc_auc_score

warnings.filterwarnings('ignore')

# Hugging Face token
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

print('Connected to warehouse')

Connected to warehouse


---
## 1. Build the feature vector

Features are aggregated per content item from a **60-day feature window (January 2026 through February 2026)** that ends before the label window (March 2026) begins. The decision moment is 2026-03-31.

- **Feature window**: 2026-01-01 to 2026-02-28 (59 days) — metrics fully knowable before March
- **Label window**: 2026-03-01 to 2026-03-31 (31 days) — outcome period

Features (5 numeric + 1 categorical):
1. `imp_feature` — total GSC impressions in the feature window
2. `pos_feature` — average GSC position in the feature window
3. `clk_feature` — total GSC clicks in the feature window
4. `days_with_imp` — distinct days with >0 impressions in the feature window
5. `content_age_days` — days since content creation (from `dim_content.content_created_date`)
6. `content_type` — categorical: keyword article / feedly article / comparison article

**Label**: `is_declining` = 1 if label-window (March) impressions < 80% of feature-window impressions.

In [2]:
# Build the full feature vector from the warehouse
feature_frame = con.sql(f"""
    WITH fea_agg AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions)  AS imp_feature,
            AVG(gsc_avg_position) AS pos_feature,
            SUM(gsc_clicks)       AS clk_feature,
            COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_with_imp
        FROM {TABLES['fact_daily']}
        WHERE report_date >= '2026-01-01' AND report_date < '2026-03-01'
        GROUP BY client_hash_id, content_hash_id
    ),
    lab_agg AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) AS imp_label
        FROM {TABLES['fact_daily']}
        WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
        GROUP BY content_hash_id
    ),
    content_meta AS (
        SELECT
            content_hash_id,
            DATEDIFF('day', content_created_date, DATE '2026-03-31') AS content_age_days,
            content_type
        FROM {TABLES['dim_content']}
        WHERE content_created_date IS NOT NULL
    )
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.imp_feature,
        f.pos_feature,
        f.clk_feature,
        f.days_with_imp,
        c.content_age_days,
        c.content_type,
        l.imp_label
    FROM fea_agg f
    JOIN lab_agg l USING (content_hash_id)
    LEFT JOIN content_meta c USING (content_hash_id)
    WHERE f.imp_feature >= 100  -- minimum volume filter
""").df()

print(f"Feature vector: {len(feature_frame):,} content items")
print(f"Columns: {list(feature_frame.columns)}")

Feature vector: 85,507 content items
Columns: ['client_hash_id', 'content_hash_id', 'imp_feature', 'pos_feature', 'clk_feature', 'days_with_imp', 'content_age_days', 'content_type', 'imp_label']


In [3]:
# Handle missing values
feature_frame['content_age_days'] = feature_frame['content_age_days'].fillna(0)
feature_frame['content_type'] = feature_frame['content_type'].fillna('unknown')

# One-hot encode content_type (safe for any model type)
content_dummies = pd.get_dummies(feature_frame['content_type'], prefix='ct_', dummy_na=False)
feature_frame = pd.concat([feature_frame, content_dummies], axis=1)
# Also keep a single code column for tree model compatibility
feature_frame['content_type_code'] = feature_frame['content_type'].astype('category').cat.codes

# Define label
feature_frame['is_declining'] = (feature_frame['imp_label'] < 0.8 * feature_frame['imp_feature']).astype(int)

# Final feature set
# All feature columns including one-hot encoded content type
content_dummy_cols = [c for c in feature_frame.columns if c.startswith('ct_')]
feature_cols = ['imp_feature', 'pos_feature', 'clk_feature', 'days_with_imp', 'content_age_days'] + content_dummy_cols

print(f"Label distribution:")
print(f"  Declining (label=1): {(feature_frame['is_declining'] == 1).sum():,} ({(feature_frame['is_declining'].mean()*100):.1f}%)")
print(f"  Stable/Up (label=0): {(feature_frame['is_declining'] == 0).sum():,} ({(1-feature_frame['is_declining'].mean())*100:.1f}%)")
print()
print(f"Content types: {feature_frame['content_type'].value_counts().to_dict()}")
print()
feature_frame[['content_hash_id'] + feature_cols + ['is_declining']].head(5)

Label distribution:
  Declining (label=1): 48,680 (56.9%)
  Stable/Up (label=0): 36,827 (43.1%)

Content types: {'keyword article': 83902, 'feedly article': 1295, 'comparison article': 310}



,content_hash_id,imp_feature,pos_feature,clk_feature,days_with_imp,content_age_days,ct__comparison article,ct__feedly article,ct__keyword article,is_declining
0,content_14c17f59aa610ab3,152.0,5.588385,2.0,3,42,False,False,True,0
1,content_15770c63daac443b,1432.0,5.419963,13.0,17,55,False,False,True,0
2,content_157db5a38382c639,806.0,6.860101,6.0,14,55,False,False,True,0
3,content_16459ed7545d1aee,231.0,7.384077,0.0,16,55,False,False,True,0
4,content_16515c58d44f1f37,128.0,13.196991,1.0,31,78,False,False,True,1


---
## 2. Feature notes

| # | Feature | Meaning | Missing? | Categorical? | Available when? |
|---|---|---|---|---|---|
| 1 | `imp_feature` | Total GSC impressions in Jan–Feb 2026 | No | No | Knowable at decision moment because January–February ended before March began |
| 2 | `pos_feature` | Average GSC position in Jan–Feb 2026 | No | No | Knowable because position data from Jan–Feb was fully recorded by end of February |
| 3 | `clk_feature` | Total GSC clicks in Jan–Feb 2026 | No | No | Knowable because all clicks from the feature window were measured before March 1st |
| 4 | `days_with_imp` | Distinct days with >0 impressions in feature window | No | No | Knowable because daily impression data from Jan–Feb is complete and finalized |
| 5 | `content_age_days` | Days since creation date (as of 2026-03-31) | Yes — 0 rows have NULL creation date (filled with 0) | No | Knowable because `content_created_date` is a fixed attribute in `dim_content` |
| 6 | `ct_*` (one-hot) | Content type dummies: keyword article, feedly article, comparison article | Yes — 'unknown' category gets all-zeros | Yes (one-hot, 3 binary columns) | Knowable because content type is a fixed metadata attribute assigned at creation |

---
## 3. The leakage hunt

I attack my own model systematically using the taxonomy from the hunting-leakage skill:

### 3a. Label-derived features test
*Train once WITH a leaky column, once WITHOUT. The score should collapse.*

The leaky column: `imp_change_pct` = (imp_label − imp_feature) / imp_feature. This is the % change from feature window to label window. The label is defined as `imp_label < 0.8 * imp_feature`, which is equivalent to `imp_change_pct < -0.2`. Including `imp_change_pct` as a feature lets the model read the label directly.

In [4]:
# --- LABEL-DERIVED FEATURES TEST ---

def train_eval(X, y, label='Model'):
    """Train a random forest and return metrics."""
    valid = X.notna().all(axis=1)
    X, y = X[valid], y[valid]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
    auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
    base = max(y_te.mean(), 1 - y_te.mean())
    print(f"{label:35s} | ROC AUC: {auc:.4f} | Base rate: {base:.3f}")
    return auc, base, model

X_clean = feature_frame[feature_cols].copy()
y = feature_frame['is_declining'].copy()

auc_honest, base_honest, _ = train_eval(X_clean, y, 'Honest (no leakage)')

# Now add the leaky column
X_leaky = X_clean.copy()
X_leaky['imp_change_pct'] = (feature_frame['imp_label'] - feature_frame['imp_feature']) / feature_frame['imp_feature']
auc_leaky, base_leaky, _ = train_eval(X_leaky, y, 'With imp_change_pct (leaky)')

print()
print(f"Key finding: AUC jumped from {auc_honest:.3f} to {auc_leaky:.3f}")
print(f"That's a {(auc_leaky - auc_honest) / max(0.001, 1 - auc_honest) * 100:.0f}% closure toward perfect.")
print()
print("Conclusion: imp_change_pct is DIRECT LEAKAGE. Removing it.")

Honest (no leakage)                 | ROC AUC: 0.7900 | Base rate: 0.569


With imp_change_pct (leaky)         | ROC AUC: 1.0000 | Base rate: 0.569

Key finding: AUC jumped from 0.790 to 1.000
That's a 100% closure toward perfect.

Conclusion: imp_change_pct is DIRECT LEAKAGE. Removing it.


### 3b. Future/overlapping windows test
*Check that no feature aggregates data from the label window (March 2026) or later.*

All six features are computed from data strictly before March 1, 2026. The only columns derived from March data are `imp_label` and `is_declining`, which are NEVER used as features.

The query table `fact_content_query_90d` was excluded from the contract because its 90-day window overlaps the feature window. We also verify here that no feature accidentally uses it.

In [5]:
# Query-verify: max feature-window date is < 2026-03-01
fea_bounds = con.sql(f"""
    SELECT
        MAX(report_date) AS max_feature_date,
        MIN(report_date) AS min_feature_date,
        COUNT(*) AS total_feature_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-01-01' AND report_date < '2026-03-01'
""").df()

lab_bounds = con.sql(f"""
    SELECT
        MIN(report_date) AS min_label_date,
        MAX(report_date) AS max_label_date
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
""").df()

max_fea = fea_bounds['max_feature_date'].iloc[0]
min_lab = lab_bounds['min_label_date'].iloc[0]
clean = max_fea < min_lab

print(f"Feature window MAX date: {max_fea}")
print(f"Label window MIN date:   {min_lab}")
print(f"No overlap (fea_max < lab_min): {clean}")
print()
print("All 6 features use only feature-window data:")
for col in ['imp_feature', 'pos_feature', 'clk_feature', 'days_with_imp', 'content_age_days', 'content_type_code']:
    print(f"  {col:<25} \u2190 feature window only, no March data")
print()
print("No features from fact_content_query_90d are used.")
print("No features from dim_clients (access profile, GA4 start date) are used.")
print()
print(f"Verdict: {'CLEAN' if clean else 'LEAKAGE FOUND'} \u2014 no future/overlapping windows.")
assert clean, 'OVERLAP FOUND: feature window extends into label window!'

Feature window MAX date: 2026-02-28 00:00:00
Label window MIN date:   2026-03-01 00:00:00
No overlap (fea_max < lab_min): True

All 6 features use only feature-window data:
  imp_feature               ← feature window only, no March data
  pos_feature               ← feature window only, no March data
  clk_feature               ← feature window only, no March data
  days_with_imp             ← feature window only, no March data
  content_age_days          ← feature window only, no March data
  content_type_code         ← feature window only, no March data

No features from fact_content_query_90d are used.
No features from dim_clients (access profile, GA4 start date) are used.

Verdict: CLEAN — no future/overlapping windows.


### 3c. Decision-derived features (product flags)
*Check that no FlyRank product decision flags or pre-computed scores entered the feature set.*

The warehouse release deliberately excludes `health_score`, `priority_score`, `action_type`, and other product decision fields. None of our six features are derived from existing product rules.

Features are built from:
- Raw observed signals: impressions, clicks, position (from `fact_content_daily_performance`)
- Content metadata: creation date, content type (from `dim_content`)

Our label `is_declining` is our own binary definition (≥20% MoM drop), not a product flag.

In [6]:
print("Product flags NOT in warehouse release: health_score, priority_score, action_type, refresh_tier")
print()
print("Our features come only from:")
print("  - fact_content_daily_performance.gsc_impressions") 
print("  - fact_content_daily_performance.gsc_avg_position")
print("  - fact_content_daily_performance.gsc_clicks")
print("  - dim_content.content_created_date")
print("  - dim_content.content_type")
print()
print("Verdict: CLEAN — no product decision flags.")

Product flags NOT in warehouse release: health_score, priority_score, action_type, refresh_tier

Our features come only from:
  - fact_content_daily_performance.gsc_impressions
  - fact_content_daily_performance.gsc_avg_position
  - fact_content_daily_performance.gsc_clicks
  - dim_content.content_created_date
  - dim_content.content_type

Verdict: CLEAN — no product decision flags.


### 3d. Random split vs grouped split (by client)
*A random split lets the model memorize client-specific quirks. The honest test is: does it work on clients it never saw?*

We compare Random Forest performance under:
- **Random split** (25% test, stratified, random state)
- **Grouped split** by `client_hash_id` (some clients held out entirely)

In [7]:
X = feature_frame[feature_cols].copy()
y = feature_frame['is_declining'].copy()
groups = feature_frame['client_hash_id'].copy()

valid = X.notna().all(axis=1)
X, y, groups = X[valid], y[valid], groups[valid]

# --- RANDOM SPLIT ---
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf_random = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr_r, y_tr_r)
auc_random = roc_auc_score(y_te_r, rf_random.predict_proba(X_te_r)[:, 1])
base_random = max(y_te_r.mean(), 1 - y_te_r.mean())

print("RANDOM SPLIT (stratified, 75/25)")
print(f"  ROC AUC:    {auc_random:.4f}")
print(f"  Base rate:  {base_random:.3f}")
print()

# --- GROUPED SPLIT ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_tr_g, X_te_g = X.iloc[train_idx], X.iloc[test_idx]
y_tr_g, y_te_g = y.iloc[train_idx], y.iloc[test_idx]

rf_grouped = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr_g, y_tr_g)
auc_grouped = roc_auc_score(y_te_g, rf_grouped.predict_proba(X_te_g)[:, 1])
base_grouped = max(y_te_g.mean(), 1 - y_te_g.mean())

print("GROUPED SPLIT (by client_hash_id, 75/25)")
print(f"  ROC AUC:    {auc_grouped:.4f}")
print(f"  Base rate:  {base_grouped:.3f}")
print(f"  Test clients: {X_te_g.shape[0]:,} rows from {groups.iloc[test_idx].nunique()} unseen clients")
print()

print("=" * 60)
print("COMPARISON")
print("=" * 60)
print(f"  {'Split':<20} {'ROC AUC':>8} {'Base rate':>10}")
print(f"  {'-'*20} {'-'*8} {'-'*10}")
print(f"  {'Random':<20} {auc_random:>8.4f} {base_random:>10.3f}")
print(f"  {'Grouped (by client)':<20} {auc_grouped:>8.4f} {base_grouped:>10.3f}")
gap = (auc_random - auc_grouped) / max(0.001, 1 - auc_grouped) * 100
print()
print(f"Gap: Random AUC - Grouped AUC = {auc_random - auc_grouped:.4f} ({gap:.0f}% closure)")
print()
if gap > 10 or (auc_random - auc_grouped) > 0.03:
    print("Finding: The random split overestimates skill by memorizing client patterns.")
    print(f"  Gap magnitude: {auc_random - auc_grouped:.4f} ({gap:.0f}% closure)")
    print("The grouped-split ROC AUC is the honest number.")
else:
    print("Finding: The gap is small — the model generalizes across clients.")

RANDOM SPLIT (stratified, 75/25)
  ROC AUC:    0.7900
  Base rate:  0.569



GROUPED SPLIT (by client_hash_id, 75/25)
  ROC AUC:    0.7065
  Base rate:  0.512
  Test clients: 9,701 rows from 9 unseen clients

COMPARISON
  Split                 ROC AUC  Base rate
  -------------------- -------- ----------
  Random                 0.7900      0.569
  Grouped (by client)    0.7065      0.512

Gap: Random AUC - Grouped AUC = 0.0835 (28% closure)

Finding: The random split overestimates skill by memorizing client patterns.
  Gap magnitude: 0.0835 (28% closure)
The grouped-split ROC AUC is the honest number.


### 3e. Feature importance sanity check
*Print top feature importances and investigate anything that looks 'too good.'*

In [8]:
importances = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_random.feature_importances_
}).sort_values('importance', ascending=False)

print("=" * 50)
print("FEATURE IMPORTANCE (random forest, random split)")
print("=" * 50)
for i, row in importances.iterrows():
    bar = '█' * int(row['importance'] * 100)
    print(f"  {row['feature']:<25} {row['importance']:.4f}  {bar}")
print()
print("Sanity check: Does any feature dominate?")
top = importances.iloc[0]
second = importances.iloc[1]['importance'] if len(importances) > 1 else 0
if top['importance'] > 3 * second:
    print(f"  WARNING: {top['feature']} (imp={top['importance']:.3f}) is >3x the next feature (imp={second:.3f})")
    print("  Investigate whether this feature accidentally leaks the label.")
else:
    n_feats = len(importances)
    print(f"  No feature dominates (top = {top['feature']} at {top['importance']:.3f}, next = {second:.3f}).")
    print(f"  Signal appears reasonably distributed across {n_feats} features (equal split would be ~{1/n_feats:.3f}).")

FEATURE IMPORTANCE (random forest, random split)
  pos_feature               0.2720  ███████████████████████████
  imp_feature               0.2453  ████████████████████████
  content_age_days          0.2308  ███████████████████████
  days_with_imp             0.1698  ████████████████
  clk_feature               0.0757  ███████
  ct__feedly article        0.0038  
  ct__keyword article       0.0019  
  ct__comparison article    0.0008  

Sanity check: Does any feature dominate?
  No feature dominates (top = pos_feature at 0.272, next = 0.245).
  Signal appears reasonably distributed across 8 features (equal split would be ~0.125).


### 3f. Attack checklist summary

| Check | Result | Evidence |
|---|---|---|
| Timeline drawn | PASS | Features from Jan–Feb, label from March |
| No label-derived features | PASS | Tested: removing `imp_change_pct` collapses AUC from near-1 to honest |
| No product flags | PASS | Warehouse has no product flags; features from raw signals only |
| Population selection check | PASS | Minimum volume filter (`imp_feature >= 100`) does not depend on outcome window |
| Split grouped by entity | DONE | Grouped split by client tested above |
| Base rate printed | DONE | Printed alongside every metric |
| Top feature sanity check | DONE | No single feature dominates |
| Metrics recomputed out-of-fold | DONE | Train/test split on all models |

---
## 4. What I excluded and why

| Excluded | Reason |
|---|---|
| `fact_content_query_90d` | Its fixed 90-day window overlaps the feature window (Dec 2025–Feb 2026). Per-content aggregates would include feature-window data, creating an ambiguous train/test boundary. |
| `ga4_data_available` flag | It tells us whether GA4 data exists for a row, but it's a property of the LABEL WINDOW (we'd be filtering rows based on whether they have analytics data in the period we're predicting). Not a feature. |
| `gsc_data_available` flag | Same logic: a data-availability flag for the prediction period is not a feature. |
| `trend_direction`, `trend_pct` | These are the SOURCE of the label in the starter dataset. Using them as features would be label-derived leakage. In the warehouse, they don't exist — but if we rebuilt them from March data, same rule applies. |
| Product decision flags (`health_score`, `priority_score`, `action_type`) | Not in the warehouse release. If they were, they'd encode existing product rules, not discoverable signal. |
| `client_hash_id`, `content_hash_id` | IDs are for grouping and joining only. Using them as features would memorize client/content identities. |
| Future-window metrics (anything after 2026-02-28) | The label is defined on March 2026 data. Any metric from March or later would leak the answer. |

---
## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.